In [ ]:
import os
import time
from fastapi import FastAPI
from pydantic import BaseModel
from typing import List
import torch
from sentence_transformers import SentenceTransformer
import torch.nn.functional as F
import uvicorn
import threading

In [ ]:
class EmbedRequest(BaseModel):
    texts: List[str]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SentenceTransformer("BAAI/bge-large-en-v1.5", device=device)

In [ ]:
model.eval()

In [ ]:
app = FastAPI()

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/embed")
def embed(request: EmbedRequest):

    embeddings = model.encode(
        request.texts,
        batch_size=32,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=False
    )

    return {"embeddings": embeddings.cpu().tolist()}

In [ ]:
def run():
    uvicorn.run(app, host="0.0.0.0", port=8000)

threading.Thread(target=run).start()

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

In [ ]:
!cloudflared tunnel --url http://localhost:8000

In [ ]:
!pkill -f uvicorn